# Latest upper-PPO curriculum: scenarios and learned behavior

This notebook automatically opens the newest `models/upper_ppo_3gnb/run_*` directory. It reconstructs the six time-aware training scenarios and connects their geometry to the actual PPO training log.

It shows scenario frequency, physical duration, topology and UE paths, training progress, representative load/bias/offset matrices, and safe-admission activity.

In [ ]:
import ast
import json
import os
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np
import pandas as pd
from IPython.display import display

from global_ppo_3gnb_env import DEFAULT_GNB_CONFIGS_3, SLICE_TYPES
from upper_agent_training_scenarios import UPPER_TRAINING_SCENARIO_BY_NAME

plt.style.use('seaborn-v0_8-whitegrid')
runs = sorted((ROOT / 'models/upper_ppo_3gnb').glob('run_*'), key=lambda p: p.stat().st_mtime)
RUN_DIR = runs[-1]
config = json.loads((RUN_DIR / 'config.json').read_text())
df = pd.read_csv(RUN_DIR / 'training_log.csv')
validation = pd.read_csv(RUN_DIR / 'validation_log.csv') if (RUN_DIR / 'validation_log.csv').exists() else pd.DataFrame()

print('Run:', RUN_DIR)
print('Training rows:', len(df), '| completed episodes:', int(df.done.sum()))
print('Configured timesteps:', config['total_timesteps'], '| final logged step:', int(df.step.max()))
display(pd.Series({
    'upper window (s)': config['upper_window_seconds'],
    'local steps / upper window': config['local_steps_per_global'],
    'local mobility step (s)': df.local_step_seconds.iloc[0],
    'radio substeps / local step': config['radio_substeps'],
    'radio service / upper window (s)': df.radio_service_seconds_per_upper_window.iloc[0],
}).to_frame('value'))


## Scenario coverage and episode timing

Each reset selects one scenario. The first plot verifies curriculum balance. The second plot shows the physical duration assigned to each scenario, not merely its number of PPO steps.

In [ ]:
episode_summary = df.groupby('episode', as_index=False).agg(
    scenario_name=('scenario_name', 'first'),
    duration_s=('episode_duration_s', 'max'),
    final_time_s=('episode_time_s', 'max'),
    episode_return=('episode_return', 'last'),
    final_variance=('load_variance', 'last'),
    total_handovers=('handover_count', 'sum'),
    completed=('done', 'max'),
)
completed = episode_summary[episode_summary.completed].copy()
counts = completed.scenario_name.value_counts().sort_index()
durations = completed.groupby('scenario_name').duration_s.first().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
counts.plot(kind='bar', ax=axes[0], color='tab:blue')
axes[0].set_title('Completed episodes per scenario')
axes[0].set_ylabel('episodes')
axes[0].tick_params(axis='x', rotation=35)
durations.plot(kind='bar', ax=axes[1], color='tab:orange')
axes[1].set_title('Physical duration of one episode')
axes[1].set_ylabel('simulated seconds')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()
display(pd.concat([counts.rename('episodes'), durations.rename('duration_s')], axis=1))


## Reconstructed scenario geometry

Triangles are gNBs, circles are UE starting groups, and arrows show their intended movement during the episode. Circle size reflects UE count; color identifies the slice.

In [ ]:
slice_colors = {'eMBB': 'tab:blue', 'URLLC': 'tab:red', 'mMTC': 'tab:green'}
gnbs = {int(cfg['id']): cfg for cfg in DEFAULT_GNB_CONFIGS_3}

def group_state(group):
    source = gnbs[group.source_gnb]
    if group.target_gnb is None or group.speed_mps <= 0:
        return source['x'], source['y'], source['x'], source['y']
    target = gnbs[group.target_gnb]
    dx, dy = target['x'] - source['x'], target['y'] - source['y']
    distance = max(np.hypot(dx, dy), 1e-9)
    px, py = -dy / distance, dx / distance
    x0 = source['x'] + group.path_progress * dx + group.lateral_offset_m * px
    y0 = source['y'] + group.path_progress * dy + group.lateral_offset_m * py
    x1 = x0 + group.speed_mps * group.duration_s * dx / distance if hasattr(group, 'duration_s') else x0
    y1 = y0 + group.speed_mps * group.duration_s * dy / distance if hasattr(group, 'duration_s') else y0
    return x0, y0, x1, y1

scenario_names = list(counts.index)
fig, axes = plt.subplots(2, 3, figsize=(17, 11))
for ax, name in zip(axes.flat, scenario_names):
    scenario = UPPER_TRAINING_SCENARIO_BY_NAME[name]
    for cfg in gnbs.values():
        ax.add_patch(Circle((cfg['x'], cfg['y']), cfg['coverage_radius'], fill=False, alpha=0.15))
        ax.scatter(cfg['x'], cfg['y'], marker='^', s=130, color='black')
        ax.text(cfg['x'] + 8, cfg['y'] + 8, f"gNB{cfg['id']}")
    for group_idx, group in enumerate(scenario.groups):
        source = gnbs[group.source_gnb]
        if group.target_gnb is None or group.speed_mps <= 0:
            angle = 2*np.pi*(group_idx+1)/max(len(scenario.groups), 1)
            x0, y0 = source['x'] + 45*np.cos(angle), source['y'] + 45*np.sin(angle)
            x1, y1 = x0, y0
        else:
            target = gnbs[group.target_gnb]
            dx, dy = target['x']-source['x'], target['y']-source['y']
            distance = max(np.hypot(dx, dy), 1e-9)
            px, py = -dy/distance, dx/distance
            x0 = source['x'] + group.path_progress*dx + group.lateral_offset_m*px
            y0 = source['y'] + group.path_progress*dy + group.lateral_offset_m*py
            x1 = x0 + group.speed_mps*scenario.duration_s*dx/distance
            y1 = y0 + group.speed_mps*scenario.duration_s*dy/distance
        ax.scatter(x0, y0, s=45*group.count, color=slice_colors[group.slice_type], alpha=0.75, edgecolor='black')
        if (x1, y1) != (x0, y0):
            ax.annotate('', xy=(x1,y1), xytext=(x0,y0), arrowprops={'arrowstyle':'->','lw':2,'color':slice_colors[group.slice_type]})
        ax.text(x0+5, y0+5, f'{group.count} {group.slice_type}\nL={group.total_load:.2f}', fontsize=8)
    ax.set_title(f'{name}\n{scenario.duration_s:.0f}s')
    ax.set_xlim(-220, 670); ax.set_ylim(-260, 650); ax.set_aspect('equal')
plt.tight_layout()
plt.show()


## Training evolution

The curves use rolling means over completed episodes. They show whether return improves, final imbalance decreases, and handover intensity remains controlled.

In [ ]:
ordered = completed.sort_values('episode').copy()
window = 60
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for column, ax, title in [
    ('episode_return', axes[0], 'Episode return'),
    ('final_variance', axes[1], 'Final load variance'),
    ('total_handovers', axes[2], 'Handovers per episode'),
]:
    ax.plot(ordered.episode, ordered[column], alpha=0.18)
    ax.plot(ordered.episode, ordered[column].rolling(window, min_periods=10).mean(), linewidth=2)
    ax.set_title(title); ax.set_xlabel('episode')
plt.tight_layout(); plt.show()

scenario_result = completed.groupby('scenario_name').agg(
    mean_return=('episode_return','mean'),
    mean_final_variance=('final_variance','mean'),
    mean_handovers=('total_handovers','mean'),
).sort_index()
display(scenario_result.round(4))


## Representative learned episode for each scenario

For each scenario, this section selects its latest completed episode and plots load trajectories, upper biases, and handovers over physical time.

In [ ]:
latest_episode = completed.groupby('scenario_name').episode.max().to_dict()
fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharex=False)
for ax, name in zip(axes.flat, scenario_names):
    episode = latest_episode[name]
    part = df[df.episode == episode]
    for g in range(3):
        for slice_type in SLICE_TYPES:
            ax.plot(part.episode_time_s, part[f'load_g{g}_{slice_type}'], alpha=0.65, label=f'g{g}-{slice_type}')
    event_times = part.loc[part.handover_count > 0, 'episode_time_s']
    for t in event_times:
        ax.axvline(t, color='black', alpha=0.15)
    ax.set_title(f'{name} | episode {episode}')
    ax.set_xlabel('simulated seconds'); ax.set_ylabel('load')
    ax.set_ylim(0, 1.05)
axes[0,0].legend(fontsize=6, ncol=3)
plt.tight_layout(); plt.show()


## Load, bias, and directional-offset matrices

The heatmaps below use the final step of the latest `three_way_slice_conflict` episode. Load and bias are 3×3 matrices. Directional offsets are shown as six directed links × three slices.

In [ ]:
matrix_scenario = 'three_way_slice_conflict'
matrix_episode = latest_episode[matrix_scenario]
row = df[df.episode == matrix_episode].iloc[-1]
load = np.asarray(json.loads(row.load_matrix), dtype=float)
bias = np.asarray(json.loads(row.bias_matrix), dtype=float)
offsets = np.asarray(json.loads(row.directional_offset_tensor), dtype=float)
directions = ['g0→g1','g0→g2','g1→g0','g1→g2','g2→g0','g2→g1']
offset_matrix = offsets.reshape(6, 3)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
items = [
    (load, axes[0], 'Load matrix', 'YlOrRd', 0, 1, [f'gNB{i}' for i in range(3)]),
    (bias, axes[1], 'Upper bias matrix', 'coolwarm', -1, 1, [f'gNB{i}' for i in range(3)]),
    (offset_matrix, axes[2], 'Directional offset matrix (dB)', 'coolwarm', -12, 6, directions),
]
for matrix, ax, title, cmap, vmin, vmax, ylabels in items:
    image = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(3), SLICE_TYPES); ax.set_yticks(range(len(ylabels)), ylabels)
    ax.set_title(title)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f'{matrix[i,j]:.2f}', ha='center', va='center', fontsize=8)
    fig.colorbar(image, ax=ax, shrink=0.75)
plt.tight_layout(); plt.show()


## Safe-admission diagnostics and validation summary

Admission statistics are stored as JSON in every training row. This cell aggregates eligible, accepted, and rejected candidates and reports the trainer's final validation metrics.

In [ ]:
stats = pd.DataFrame([json.loads(value) for value in df.safe_admission_stats])
stats['scenario_name'] = df.scenario_name.values
admission_by_scenario = stats.groupby('scenario_name').sum(numeric_only=True)
display(admission_by_scenario)

admission_by_scenario[['eligible','accepted','rejected_capacity','rejected_no_pressure','rejected_target_safety']].plot(
    kind='bar', figsize=(15,5), logy=True
)
plt.ylabel('candidate count (log scale)')
plt.title('Safe-admission outcomes by scenario')
plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()

display(pd.Series(config.get('validation', {})).to_frame('value'))
assert set(counts.index) == set(UPPER_TRAINING_SCENARIO_BY_NAME)
assert completed.final_time_s.eq(completed.duration_s).all()
assert df.step.max() >= config['total_timesteps']
print('PASS: all curriculum scenarios appear, completed episodes reach their configured duration, and training reached its target.')
